In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import json
import subprocess
import sys
from pathlib import Path
from IPython.display import display
from pyspex.spex import Session


In [2]:
# --- 1. SELECT DATASET + CONVERT VIA CLI ---
obsid = "0865600201"
instrument = "rgs"   # "pn" or "rgs"
base_interval = "Persistent"
use_grouped = True
overwrite_conversion = False
# Optional global fit-iteration cap for SPEX stages. Set to None for no fixed cap.
FIT_ITER_CAP = None
# Thread override for SPEX startup. Set to None to keep SPEX auto-selection.
SPEX_THREADS = 4

# Blind-search runtime controls (quick-look defaults; tighten for final runs).
BLIND_SEARCH_RUN = False         # Set False to skip the blind search and hide its panel
BLIND_SEARCH_DLAM = 0.05        # Use 0.01 for full-resolution scans
BLIND_SEARCH_MAX_POINTS = 120   # None = no cap
BLIND_SEARCH_ITER_CAP = 8       # Smaller = faster, less precise. None for full convergence.
BLIND_SEARCH_REFIT_BASELINE = False # Whether to refit the baseline model at each line-search step
BLIND_SEARCH_MAKE_PLOT = False  # Script-local diagnostic PNG; notebook plot is handled below

# Thermal continuum component to test (e.g., "bb" or "dbb").
THERMAL_MODEL = "dbb"
THERMAL_LABEL = THERMAL_MODEL
THERMAL_KT_ALIASES = ["t", "kT", "tin", "ktin", "kt"]
THERMAL_NORM_ALIASES = ["norm", "N", "nd"]

# Display format for numerical values: True = scientific notation, False = normal notation
USE_SCIENTIFIC_NOTATION = True

# This notebook lives in notebooks/, so repo root is one level up.
repo_root = Path("..").resolve()
converter = repo_root / "scripts" / "spex_port" / "port_to_spex.py"


def _out_base(root: Path, obs: str, inst: str, interval: str, grouped: bool) -> Path:
    suffix = f"{interval}_grp" if grouped else interval
    return root / "products" / obs / inst / "spex" / f"{inst}_{suffix}_spex"


if instrument not in {"pn", "rgs"}:
    raise ValueError("instrument must be 'pn' or 'rgs'")

spex_out_base = _out_base(repo_root, obsid, instrument, base_interval, use_grouped)
cmd = [
    sys.executable,
    str(converter),
    instrument,
    "--obsid",
    obsid,
    "--interval",
    base_interval,
    "--out-base",
    str(spex_out_base),
]
if use_grouped:
    cmd.append("--grouped")
if overwrite_conversion:
    cmd.append("--overwrite")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=repo_root)

manifest_path = spex_out_base.with_suffix(".manifest.json")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print("Loaded manifest:", manifest_path)
print("SPEX outputs:", manifest["outputs"])

# Save all fit artifacts in one designated SPEX folder (no run-archive folders).
fit_artifact_dir = spex_out_base.parent
fit_artifact_dir.mkdir(parents=True, exist_ok=True)
fit_tag = f"{instrument}_{base_interval}{'_grp' if use_grouped else ''}"

# Organize saved outputs into dedicated subfolders.
artifact_dirs = {
    "logs": fit_artifact_dir / "logs",
    "tables": fit_artifact_dir / "fit_tables",
    "summaries": fit_artifact_dir / "summaries",
    "line_search": fit_artifact_dir / "line_search",
    "plots": fit_artifact_dir / "plots",
}
for _p in artifact_dirs.values():
    _p.mkdir(parents=True, exist_ok=True)

print("Fit artifact directory:", fit_artifact_dir)


Running: /home/kyle/anaconda3/envs/spex/bin/python /media/kyle/kyle_phd/Swift-j1858.6-0814/scripts/spex_port/port_to_spex.py rgs --obsid 0865600201 --interval Persistent --out-base /media/kyle/kyle_phd/Swift-j1858.6-0814/products/0865600201/rgs/spex/rgs_Persistent_grp_spex --grouped
Read source PHA spectrum... OK
Read background PHA spectrum... OK
Read RMF response matrix... 

/home/kyle/anaconda3/envs/spex/lib/python3.11/site-packages/pyspextools/io/rmf.py:168: RuntimeWarning: overflow encountered in scalar add
  fgroup = fgroup + self.NumberGroups[i]


OK
Check OGIP source spectrum... OK
Check OGIP background spectrum... OK
Check OGIP response matrix... OK
Convert OGIP spectra to spo format... OK
Convert OGIP response to res format... 

/home/kyle/anaconda3/envs/spex/lib/python3.11/site-packages/pyspextools/io/convert.py:95: RuntimeWarning: invalid value encountered in scalar divide
  spo.ochan[i] = spo.ochan[i] - back.Rate[i] * fb / back.AreaScaling[i]
/home/kyle/anaconda3/envs/spex/lib/python3.11/site-packages/pyspextools/io/convert.py:96: RuntimeWarning: invalid value encountered in scalar divide
  spo.dochan[i] = spo.dochan[i] + (back.StatError[i] * fb / back.AreaScaling[i]) ** 2
/home/kyle/anaconda3/envs/spex/lib/python3.11/site-packages/pyspextools/io/convert.py:97: RuntimeWarning: invalid value encountered in scalar divide
  spo.mbchan[i] = back.Rate[i] * fb / back.AreaScaling[i]
/home/kyle/anaconda3/envs/spex/lib/python3.11/site-packages/pyspextools/io/convert.py:98: RuntimeWarning: invalid value encountered in scalar divide
  spo.dbchan[i] = (back.StatError[i] * fb / back.AreaScaling[i]) ** 2


OK
Check OGIP source spectrum... OK
Check OGIP background spectrum... OK
Check OGIP response matrix... OK
Convert OGIP spectra to spo format... OK
Convert OGIP response to res format... OK
Read source PHA spectrum... OK
Read background PHA spectrum... OK
Read RMF response matrix... OK
Check OGIP source spectrum... OK
Check OGIP background spectrum... OK
Check OGIP response matrix... OK
Convert OGIP spectra to spo format... OK
Convert OGIP response to res format... OK
Check OGIP source spectrum... OK
Check OGIP background spectrum... OK
Check OGIP response matrix... OK
Convert OGIP spectra to spo format... OK
Convert OGIP response to res format... OK
Read source PHA spectrum... OK
Read background PHA spectrum... OK
Read RMF response matrix... OK
Check OGIP source spectrum... OK
Check OGIP background spectrum... OK
Check OGIP response matrix... OK
Convert OGIP spectra to spo format... OK
Convert OGIP response to res format... OK
Check OGIP source spectrum... OK
Check OGIP background spec

In [3]:
# --- 3. INITIALIZE SPEX & FILTER DATA ---

# SPEX reads OMP_NUM_THREADS at startup, so set this before creating Session().
if SPEX_THREADS is not None:
    os.environ["OMP_NUM_THREADS"] = str(int(SPEX_THREADS))
    print(f"Using OMP_NUM_THREADS={os.environ['OMP_NUM_THREADS']}")

# Initialize the SPEX session
s = Session()


# Load the newly converted .res and .spo files
s.data(str(spex_out_base.with_suffix(".res")), str(spex_out_base.with_suffix(".spo")))

# Apply the energy filters from the original XSPEC script (1.0 to 7.5 keV)
# SPEX syntax: ignore(instrument, region, lower_limit, upper_limit, unit)
if instrument == "pn":
    s.ignore(1, 1, 0.0, 0.8, "kev")
    s.ignore(1, 1, 7.0, 100.0, "kev")

Using OMP_NUM_THREADS=16
 Welcome kyle to SPEX version 3.08.02

 NEW in this version of SPEX: 
07-10-2025 Added plot component functionality
07-10-2025 Added atbl model to load Xspec table models
07-10-2025 Added magnetism module for pion
07-10-2025 Added ebit model for laboratory astrophysics
07-10-2025 Improvement of partial covering factor
07-10-2025 Update of quick calculation mode

 OMP_NUM_THREADS is set to 16 threads.

 Currently using SPEXACT version 2.07.00. Type `help var calc` for details.


In [4]:
# --- 4. MODEL DEFINITION ---
def _set_par_alias(comp_idx, aliases, value, thaw=None):
    """Try parameter aliases to stay robust across pyspex builds."""
    for pname in aliases:
        try:
            if thaw is None:
                s.par(1, comp_idx, pname, value)
            else:
                s.par(1, comp_idx, pname, value, thawn=thaw)
            return pname
        except Exception:
            continue
    print(f"Warning: could not set component {comp_idx} with aliases {aliases}")
    return None


def _get_par_alias(comp_idx, aliases):
    """Return first matching Parameter object and its alias name."""
    for pname in aliases:
        try:
            p = s.par_get(1, comp_idx, pname)
        except Exception:
            p = None
        if p is not None:
            return p, pname
    return None, None


def _display_sci(df, float_cols=None, sig=6, scientific=None):
    """Display selected DataFrame columns in scientific or normal notation.

    Parameters:
    -----------
    df : DataFrame
        Input dataframe to display
    float_cols : list, optional
        Columns to format. If None, all numeric columns are formatted.
    sig : int, default=6
        Number of significant figures
    scientific : bool, optional
        If True, use scientific notation; if False, use normal notation.
        If None (default), use the global USE_SCIENTIFIC_NOTATION setting.
    """
    out = df.copy()
    if float_cols is None:
        float_cols = out.select_dtypes(include=[np.number]).columns.tolist()

    use_sci = scientific if scientific is not None else USE_SCIENTIFIC_NOTATION

    for col in float_cols:
        if use_sci:
            out[col] = out[col].apply(
                lambda x: f"{float(x):.{sig}e}" if pd.notna(x) else "nan"
            )
        else:
            out[col] = out[col].apply(
                lambda x: f"{float(x):.{sig}g}" if pd.notna(x) else "nan"
            )
    display(out)


def _par_unit(par):
    """Best-effort unit lookup across pyspex builds."""
    for attr in ("unit", "units", "u", "unit_name"):
        try:
            val = getattr(par, attr)
        except Exception:
            continue
        if val not in (None, ""):
            return str(val)
    return "-"


def _safe_float(x, default=np.nan):
    try:
        v = float(x)
    except Exception:
        return default
    return v if np.isfinite(v) else default


def _fit_stage(label, niter=None):
    """Run one SPEX fit stage; optionally apply an iteration cap."""
    if niter is None:
        print(f"{label} (iter cap: default/none)")
    else:
        print(f"{label} (max iter={int(niter)})")
        s.command(f"fit iter {int(niter)}")
    s.fit()
    cstat_now = _safe_float(getattr(s.opt_fit, "cstat", np.nan))
    if np.isfinite(cstat_now):
        print(f"  -> cstat: {cstat_now:.6e}")
    else:
        print("  -> cstat: nan")
    return cstat_now


def _unit_lookup_from_free_df(df):
    """Build (component, parameter)->unit lookup from par_show_free output when possible."""
    if df is None or len(df) == 0:
        return {}

    # Try common column naming variants across pyspex versions.
    lc = {str(c).lower(): c for c in df.columns}
    comp_col = lc.get("comp") or lc.get("component") or lc.get("icomp")
    par_col = lc.get("param") or lc.get("parameter") or lc.get("name")
    unit_col = lc.get("unit") or lc.get("units")
    if comp_col is None or par_col is None or unit_col is None:
        return {}

    lookup = {}
    for _, r in df.iterrows():
        ci = _safe_float(r.get(comp_col), default=np.nan)
        if not np.isfinite(ci):
            continue
        pname = str(r.get(par_col, "")).strip().lower()
        unit = str(r.get(unit_col, "")).strip()
        if pname == "" or unit == "":
            continue
        lookup[(int(ci), pname)] = unit
    return lookup


_EXPECTED_UNITS = {
    "nh": "1E28/m**2",
    "nH": "1E28/m**2",
    "n": "1E28/m**2",
    "xil": "1E-9 W m",
    "logxi": "-",
    "xi": "1E-9 W m",
    "fcov": "-",
    "cov": "-",
    "cover": "-",
    "f": "-",
    "frac": "-",
    "t": "keV",
    "kT": "keV",
    "tin": "keV",
    "ktin": "keV",
    "kt": "keV",
    "gamm": "-",
    "gamma": "-",
    "gam": "-",
    "phoindex": "-",
    "norm": "-",
    "N": "-",
    "nd": "-",
}

# Model requested: hot(galactic)*xabs(ionized)*(thermal+pow)

s.com("hot")   # Component 1: Galactic absorption
s.com("xabs")  # Component 2: Local ionized absorption
s.com(THERMAL_MODEL)  # Component 3: Thermal continuum
s.com("pow")   # Component 4: Power law

rel_array = np.array([1, 2], dtype=int)
s.com_rel(1, 3, rel_array)
s.com_rel(1, 4, rel_array)

# User-requested Galactic NH fixed value (XSPEC-style: 0.2 in units of 1e22 cm^-2)
NH_GAL = 0.002

# Galactic absorber fixed
_set_par_alias(1, ["t", "kT"], 8e-4, thaw=False)
_set_par_alias(1, ["nh", "nH"], NH_GAL, thaw=False)

s.calc()


 You have defined    1 component.
 You have defined    2 components.
 You have defined    3 components.
 You have defined    4 components.


In [5]:
# --- 5. PARAMETER SETUP & FITTING ---

# Set up logging for SPEX output
fit_log_file = artifact_dirs["logs"] / f"spex_output_{fit_tag}.log"
if fit_log_file.with_suffix(fit_log_file.suffix + ".out").exists():
    fit_log_file.with_suffix(fit_log_file.suffix + ".out").unlink()
if fit_log_file.exists():
    fit_log_file.unlink()

# SPEX log out command
s.command(f"log out {fit_log_file}")

# Ionized absorber (component 2): start constrained, then thaw in stages.
_set_par_alias(2, ["nh", "nH", "n"], 0.003, thaw=False)
_set_par_alias(2, ["xil", "logxi", "logx", "xi"], 2.0, thaw=False)
_set_par_alias(2, ["fcov", "cov", "cover", "f", "frac"], 0.7, thaw=False)

# Thermal continuum (component 3): start lower to avoid soft overshoot.
_set_par_alias(3, THERMAL_KT_ALIASES, 0.5, thaw=True)
_set_par_alias(3, THERMAL_NORM_ALIASES, 1.0e-3, thaw=False)

# Power law (component 4): start harder and let it fit the high-energy tail.
_set_par_alias(4, ["gamm", "gamma", "gam", "phoindex"], 1.6, thaw=True)
_set_par_alias(4, ["norm", "N"], 2.0e-2, thaw=True)

# Set the fit statistic to C-stat
s.command('fit stat cstat')

# Stage 0: fit continuum + Galactic absorber with ionized absorber and thermal norm frozen.
_fit_stage("Stage 0: xabs frozen, thermal norm frozen", niter=FIT_ITER_CAP)

# Stage 0b: thaw thermal normalization after hard-tail is anchored.
_set_par_alias(3, THERMAL_NORM_ALIASES, 1.0e-3, thaw=True)
_fit_stage("Stage 0b: thaw thermal norm", niter=FIT_ITER_CAP)

# Stage 1: thaw xabs NH.
_set_par_alias(2, ["nh", "nH", "n"], 0.003, thaw=True)
_fit_stage("Stage 1: thaw xabs NH", niter=FIT_ITER_CAP)

# Stage 2: thaw xabs ionization only if NH is meaningfully > 0.
xabs_nh_par, _ = _get_par_alias(2, ["nh", "nH", "n"])
xabs_nh_val = _safe_float(getattr(xabs_nh_par, "value", np.nan))
if np.isfinite(xabs_nh_val) and xabs_nh_val > 1.0e-6:
    _set_par_alias(2, ["xil", "logxi", "logx", "xi"], 2.0, thaw=True)
    _fit_stage("Stage 2: thaw xabs logxi", niter=FIT_ITER_CAP)
else:
    print(f"Stage 2 skipped: xabs NH ~ 0 ({xabs_nh_val:.3e}); keeping logxi frozen")

# Stage 3: thaw xabs covering fraction last only if xabs is active.
xabs_nh_par, _ = _get_par_alias(2, ["nh", "nH", "n"])
xabs_nh_val = _safe_float(getattr(xabs_nh_par, "value", np.nan))
if np.isfinite(xabs_nh_val) and xabs_nh_val > 1.0e-6:
    _set_par_alias(2, ["fcov", "cov", "cover", "f", "frac"], 0.7, thaw=True)
    _fit_stage("Stage 3: thaw xabs covering fraction", niter=FIT_ITER_CAP)
else:
    print(f"Stage 3 skipped: xabs NH ~ 0 ({xabs_nh_val:.3e}); keeping fcov frozen")

s.command("log close out")

# --- 5b. FIT ASSESSMENT + KEY PARAMETER TABLE ---
fit_rows = []
for stat_name, attr_name in [("C-stat", "cstat"), ("Chi-squared", "chisq"), ("W-stat", "wstat")]:
    try:
        fit_rows.append({"Statistic": stat_name, "Value": float(getattr(s.opt_fit, attr_name))})
    except Exception:
        fit_rows.append({"Statistic": stat_name, "Value": np.nan})

try:
    fit_rows.append({"Statistic": "N free parameters", "Value": int(s.opt_fit.nfree)})
except Exception:
    pass

fit_df = pd.DataFrame(fit_rows)
print("\nFit statistics:")
_display_sci(fit_df, float_cols=["Value"], sig=6)

# Build unit lookup from SPEX parameter table
free_df = None
unit_lookup = {}
try:
    free_tab = s.par_show_free()
    free_df = free_tab.to_pandas()
    unit_lookup = _unit_lookup_from_free_df(free_df)
except Exception:
    print("Note: Could not retrieve full free-parameter table with s.par_show_free().")

important = [
    ("gal_hot", 1, ["nh", "nH"]),
    ("xabs", 2, ["nh", "nH", "n"]),
    ("xabs", 2, ["xil", "logxi", "logx", "xi"]),
    ("xabs", 2, ["fcov", "cov", "cover", "f", "frac"]),
    (THERMAL_LABEL, 3, THERMAL_KT_ALIASES),
    ("pow", 4, ["gamm", "gamma", "gam", "phoindex"]),
    ("pow", 4, ["norm", "N"]),
]

rows = []
for comp_label, comp_idx, aliases in important:
    par, used_alias = _get_par_alias(comp_idx, aliases)
    if par is None:
        rows.append(
            {
                "Component": comp_label,
                "Comp#": comp_idx,
                "Parameter": aliases[0],
                "MatchedName": "-",
                "Unit": "-",
                "Value": np.nan,
                "ErrLow": np.nan,
                "ErrHigh": np.nan,
                "Status": "not found",
            }
        )
        continue

    rows.append(
        {
            "Component": comp_label,
            "Comp#": comp_idx,
            "Parameter": aliases[0],
            "MatchedName": used_alias,
            "Unit": (
                _par_unit(par)
                if _par_unit(par) != "-"
                else unit_lookup.get((comp_idx, str(used_alias).lower()), _EXPECTED_UNITS.get(aliases[0], "-"))
            ),
            "Value": float(par.value),
            "ErrLow": float(par.err_low),
            "ErrHigh": float(par.err_upp),
            "Status": "free" if bool(par.free) else "frozen",
        }
    )

param_df = pd.DataFrame(rows)
print("Key parameter summary:")
_display_sci(param_df, float_cols=["Value", "ErrLow", "ErrHigh"], sig=6)

# Optional: Show all free parameters from SPEX if you want more detail (uncomment next line)
# if free_df is not None:
#     print("\nAll free parameters (SPEX):")
#     display(free_df)

# Basic quality flags for quick triage before deeper interpretation.
quality_flags = []
try:
    cstat_val = float(getattr(s.opt_fit, "cstat"))
    if not np.isfinite(cstat_val):
        quality_flags.append("non-finite cstat")
except Exception:
    quality_flags.append("missing cstat")

if (param_df["Status"] == "not found").any():
    quality_flags.append("some key parameters not found in this SPEX build")

if param_df["Value"].isna().any():
    quality_flags.append("some key parameter values are NaN")

print("Quality flags:", quality_flags if quality_flags else ["none"])

# Persist fit tables in the designated SPEX output folder.
fit_stats_csv = artifact_dirs["tables"] / f"fit_statistics_{fit_tag}.csv"
param_csv = artifact_dirs["tables"] / f"key_parameters_{fit_tag}.csv"
fit_summary_json = artifact_dirs["summaries"] / f"fit_summary_{fit_tag}.json"

fit_df.to_csv(fit_stats_csv, index=False)
param_df.to_csv(param_csv, index=False)
fit_summary_json.write_text(
    json.dumps(
        {
            "obsid": obsid,
            "instrument": instrument,
            "interval": base_interval,
            "grouped": use_grouped,
            "manifest": str(manifest_path),
            "model": f"hot*xabs*({THERMAL_MODEL}+pow)",
            "NH_GAL": NH_GAL,
            "quality_flags": quality_flags,
        },
        indent=2,
    ),
    encoding="utf-8",
)
print("Saved fit tables:")
print(" -", fit_stats_csv)
print(" -", param_csv)
print(" -", fit_summary_json)

Stage 0: xabs frozen, thermal norm frozen (iter cap: default/none)
 fit iter  100                                                   
   114866.              4  0.500      2.000E-02   1.60
 You cannot plot this frame since nothing is defined
 
   108267.              8  0.347       41.2      -10.0
 You cannot plot this frame since nothing is defined
 
   5559.62             12  0.329      0.263      -10.0
 You cannot plot this frame since nothing is defined
 
   3126.00             16  0.291      0.227      -10.0
 You cannot plot this frame since nothing is defined
 
   3057.25             20  0.286      0.296      -10.0
 You cannot plot this frame since nothing is defined
 
   3052.18             24  0.284      0.324      -10.0
 You cannot plot this frame since nothing is defined
 
   3051.85             28  0.284      0.330      -10.0
 You cannot plot this frame since nothing is defined
 
   3051.83             32  0.284      0.331      -10.0
 You cannot plot this frame since n

,Statistic,Value
0,C-stat,3.864585e+02
1,Chi-squared,9.955871e-07
2,W-stat,3.833975e+02
3,N free parameters,3.380000e+02


---------------------------------------------------------------------------------------------------
sect comp mod  acro parameter with unit      value     status    minimum   maximum lsec lcom lpar


   1    2 xabs nh   X-Column (1E28/m**2)     0.00000   thawn         0.0   1.0E+20           

   1    3 dbb  norm Area (1E16 m**2)         549.370   thawn         0.0   1.0E+20           
   1    3 dbb  t    Temperature (keV)      0.0579613   thawn     0.00010   1.0E+03           

   1    4 pow  norm Norm (1E44 ph/s/keV)     950.453   thawn         0.0   1.0E+20           
   1    4 pow  gamm Photon index             1.37867   thawn        -10.       10.           


Key parameter summary:


,Component,Comp#,Parameter,MatchedName,Unit,Value,ErrLow,ErrHigh,Status
0,gal_hot,1,nh,nh,1E28/m**2,2.000000e-03,0.000000e+00,0.000000e+00,frozen
1,xabs,2,nh,nh,1E28/m**2,0.000000e+00,0.000000e+00,0.000000e+00,free
2,xabs,2,xil,xil,1E-9 W m,2.000000e+00,0.000000e+00,0.000000e+00,frozen
3,xabs,2,fcov,fcov,-,7.000000e-01,0.000000e+00,0.000000e+00,frozen
4,dbb,3,t,t,keV,5.796131e-02,1.446948e-02,1.446948e-02,free
5,pow,4,gamm,gamm,-,1.378669e+00,8.638923e-02,8.638923e-02,free
6,pow,4,norm,norm,-,9.504526e+02,3.016335e+01,3.016335e+01,free


Quality flags: ['none']
Saved fit tables:
 - /media/kyle/kyle_phd/Swift-j1858.6-0814/products/0865600201/rgs/spex/fit_tables/fit_statistics_rgs_Persistent_grp.csv
 - /media/kyle/kyle_phd/Swift-j1858.6-0814/products/0865600201/rgs/spex/fit_tables/key_parameters_rgs_Persistent_grp.csv
 - /media/kyle/kyle_phd/Swift-j1858.6-0814/products/0865600201/rgs/spex/summaries/fit_summary_rgs_Persistent_grp.json


In [ ]:
# --- 5c. BLIND LINE SEARCH (PN-STYLE PORT TO SPEX) ---
if BLIND_SEARCH_RUN:
    from scripts.spex_blind_search import run_blind_line_search
    line_search_result = run_blind_line_search(
        s,
        artifact_dir=artifact_dirs["line_search"],
        fit_tag=fit_tag,
        energy_min_keV=0.7,
        energy_max_keV=7.0,
        d_lambda_angstrom=BLIND_SEARCH_DLAM,
        velocity_width_km_s=1000.0,
        fit_stat="cstat",
        fit_iter_cap=BLIND_SEARCH_ITER_CAP,
        refit_baseline=BLIND_SEARCH_REFIT_BASELINE,
        make_plot=BLIND_SEARCH_MAKE_PLOT,
        max_grid_points=BLIND_SEARCH_MAX_POINTS,
        enforce_single_free_param=True,
    )
    line_search_df = line_search_result.dataframe
    print("Line-search CSV:", line_search_result.csv_path)
    print("Line-search PNG:", line_search_result.plot_path)
    display(line_search_df.head())
else:
    line_search_result = None
    line_search_df = None
    print("Blind search disabled: skipping scan and omitting the third plot panel.")


In [ ]:
# --- 6. PLOTTING ---
%matplotlib inline
# Get the SPEX plot manager
mgr = s.plot_data(ylog=True)
plot_blind_search = BLIND_SEARCH_RUN and (line_search_df is not None)
if plot_blind_search:
    # Create the figure with a data/model panel, residual panel, and blind-search panel
    fig, (ax1, ax2, ax3) = plt.subplots(
        3, 1,
        figsize=(16, 9),
        sharex=True,
        gridspec_kw={"height_ratios": [2.2, 1.0, 1.3], "hspace": 0.0},
    )
else:
    # If blind search is disabled, keep the plot to two panels.
    fig, (ax1, ax2) = plt.subplots(
        2, 1,
        figsize=(16, 8),
        sharex=True,
        gridspec_kw={"height_ratios": [2.2, 1.0], "hspace": 0.0},
    )
# Plot data + model on the top axis
# In your pyspex build, plot_data requires the target axis
mgr.plot_data(ax1)
# Inspect labels once (helpful to confirm which line is background)
for i, ln in enumerate(ax1.lines):
    print(i, repr(ln.get_label()))
# Style background spectral curve(s)
bg_lines = [
    ln for ln in ax1.lines
    if any(k in ln.get_label().lower() for k in ["bkg", "back", "background"])
]
# Fallback if labels are empty in your pyspex build:
# pick the lowest-average line as likely background
if not bg_lines and ax1.lines:
    bg_lines = [min(ax1.lines, key=lambda ln: np.nanmean(ln.get_ydata()))]
for ln in bg_lines:
    ln.set_linestyle("-")
    ln.set_drawstyle("steps-mid")
    ln.set_linewidth(1.4)
    ln.set_color("tab:blue")
    ln.set_alpha(0.9)
# Optional: hide background instead
# for ln in bg_lines:
#     ln.set_visible(False)
# Plot chi / residuals on the bottom axis
# Different pyspex versions expose this slightly differently
if hasattr(mgr, "plot_chi"):
    try:
        mgr.plot_chi(ax2)
    except TypeError:
        # Some versions may have a different call signature
        mgr.chiplot(ax2)
elif hasattr(mgr, "chiplot"):
    mgr.chiplot(ax2)
# Styling
ax1.set_yscale("log")
ax1.set_ylabel("Counts s$^{-1}$ keV$^{-1}$")
ax1.set_title(f"SPEX fit: {instrument} ({base_interval}{'_grp' if use_grouped else ''})")
ax2.set_ylabel("Residual / Chi")
ax2.axhline(0,color="r", linestyle="-")
# add grid
ax2.grid(True, which="both", ls="--", lw=0.5, alpha=0.7)
if plot_blind_search:
    # Bottom panel: PN-style blind line-search statistic
    ls_df = line_search_df.sort_values("E_keV").copy()
    if "Signed_Delta_Stat" not in ls_df.columns:
        ls_df["Signed_Delta_Stat"] = np.abs(ls_df["Delta_Stat"]) * np.sign(ls_df["Norm"])
    ax3.plot(
        ls_df["E_keV"],
        ls_df["Signed_Delta_Stat"],
        color="red",
        lw=2.0,
        label=r"Signed $\Delta$stat",
    )
    ax3.axhline(0, color="black", linestyle="--", lw=1.0)
    ax3.axhline(9, color="blue", linestyle=":", lw=1.0, label=r"~3$\sigma$ (stat=9)")
    ax3.axhline(-9, color="blue", linestyle=":", lw=1.0, alpha=0.7)
    ax3.set_ylabel(r"Signed $\Delta$stat")
    ax3.set_xlabel("Energy (keV)")
    ax3.set_xlim(0.5, 7.0)
    ax3.grid(True, which="both", ls="--", lw=0.5, alpha=0.7)
    ax3.legend(loc="best")
else:
    ax2.set_xlabel("Energy (keV)")
plt.tight_layout()
plot_png = artifact_dirs["plots"] / f"fit_plot_{fit_tag}.png"
fig.savefig(plot_png, dpi=220, bbox_inches="tight")
print("Saved plot:", plot_png)
plt.show()
